# Image extender bench - one photo at 4:3 and 16:9, every local model

**Author**: Konrad Jelen (kj) <br>
**Date**: 2026-10-25 <br>
**Source**: `data/interim/henryk-urodziny-2026/IMG_8564.png`, 962 x 1278 px <br>
**Extenders**: FLUX.1 Fill dev, Qwen-Image + InstantX Inpainting, SDXL Inpainting 0.1, FLUX.1 Kontext dev, Qwen-Image-Edit 2509, Qwen-Image-Edit 2511, OmniGen2, FLUX 3 Image Klein + Outpaint LoRA <br>
**Outputs**: `tmp/media/extended/` (bench only; a production run writes into `data/`)

The whole-frame image-to-video model redraws whatever frame it is given, so the 4:3 landscape frame must exist before that model runs and must contain no scene the video model invented. This notebook extends the portrait to a 4:3 frame and to the harder 16:9 frame with every viable local extender and lays the results out as one comparison sheet per frame so the models can be compared directly. Every tile is checked by a mechanical no-new-people gate, and its verdict is recorded on the tile and in the contact sheet.

## Approach

1. Pin the RTX PRO 6000 by UUID and start a ComfyUI server on it with its own input, output and temp folders
2. Extend `IMG_8564.png` to 1280 x 960 (4:3) and 1280 x 720 (16:9) with each extender: scale the photo to the frame height, pad the two sides with flat grey, run the model, tone-match the render to the photo, paste the scaled photo's own pixels back and crop the stride padding
3. Build one comparison sheet per frame, one tile per extender, with the source photo in the corner

Upscaling is deferred by the Star Colonel's order of 2026-10-25 06:00: the bench extends only, and the frames can be raised later.

## Output

- `tmp/media/extended/4x3/<extender>/IMG_8564.png` and `tmp/media/extended/16x9/<extender>/IMG_8564.png`, each with a `.md` sidecar
- `tmp/media/extended/grid-4x3.png` and `tmp/media/extended/grid-16x9.png`, each with a `.md` index naming every tile


## GPU Selection

Pins the RTX PRO 6000 Blackwell (nvidia-smi index 1, 96 GB) by UUID before the project package imports torch. The ComfyUI server in the setup file carries the same UUID, so every graph runs on that card. The Configuration render below shows the device the server reports.

In [ ]:
import os

# Deterministic ordering so a UUID always names the same physical card
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
# RTX PRO 6000 Blackwell 96 GB, pinned by UUID from `nvidia-smi -L`
os.environ["CUDA_VISIBLE_DEVICES"] = "GPU-a44e514a-aa5b-2844-35f1-4be28e1921e5"

## Imports

All imports in one cell. The project package comes after the GPU pin because `henryk_media.imaging` imports torch through `animation.common`. The notebook calls only the module: the module owns the graphs, the tone match, the paste-back and the sidecars.

In [ ]:
%load_ext autoreload
%autoreload 2

# Standard library
import os  # Relative paths for the images shown below

# Notebook display
from IPython.display import Image, Markdown, display  # Images by link only, never embedded

# Numerics and images
from PIL import Image as PilImage
# Rich console output
from rich import box
from rich import print as rprint
from rich.progress import (
    BarColumn,
    Progress,
    SpinnerColumn,
    TextColumn,
    TimeElapsedColumn,
)
from rich.table import Table

# Project package
from henryk_media.animation import common
from henryk_media.config import INTERIM_DATA_DIR, PROJ_ROOT
from henryk_media.imaging import composite, grids, registry
from henryk_media.imaging.extender import extend
from henryk_media.imaging.gate import PersonGate
from henryk_media.imaging.provenance import kept_strangers, load_extend
from henryk_media.imaging.server import ComfyServer
from henryk_media.imaging.setup import Setup


## Reproducibility

Seeds Python, NumPy and PyTorch in this kernel. The graphs' only random input is the sampler seed, fixed at 42 and recorded in every sidecar and checkpoint, and every model renders the same seed for the same photo, so a difference between two tiles comes from the model, not from noise.

In [ ]:
common.set_seed(42)

## Configuration

The source, both aspect targets, the prompts and the server. Every extender gets the same prompt, the same negative prompt and the same seed, so the comparison isolates the model. The cell ends by starting the server, so its render can show the device the server reports.

- **Frame** - 1280 px wide; the height follows the aspect and the photo is scaled to that height
- **4:3** - 1280 x 960, 278 px added on the left and 279 on the right, 43.5 percent new scene
- **16:9** - 1280 x 720, 369 px added on each side, 57.7 percent new scene, the harder frame
- **Padding** - `ImagePadForOutpaint` fills the new sides with flat grey #808080, the canvas every prompt-padded model in this bench expects
- **Gate** - every tile runs `face_yolov8n.pt` and the COCO person detector `yolo11n.pt`; a face or person box wholly inside an added panel is a stranger and rejects the tile, a face box crossing the photo's edge is a continuation of someone the photo shows and is recorded without rejecting the tile, and a rejected tile is re-rendered with the next seed, 42 to 47
- **Upscaling** - deferred by the Star Colonel's order of 2026-10-25 06:00; this run extends only


In [ ]:
# Input
PHOTO_SET = "henryk-urodziny-2026"
SOURCE = INTERIM_DATA_DIR / PHOTO_SET / "IMG_8564.png"  # 962 x 1278 px, ratio 0.7527
SOURCE_SIZE = (962, 1278)  # width, height of the working copy, checked in Data Loading

# Output
EXTENDED_DIR = PROJ_ROOT / "tmp/media/extended"  # bench only; a production run writes into data/
ASPECTS = {"4x3": 4 / 3, "16x9": 16 / 9}  # frame name -> width / height
SEED = 42  # one sampler seed for every model and both frames

# Prompts, identical for every extender. The opening sentence is the ausboss outpaint LoRA's
# trigger, so the FLUX 3 route and every other model read the same instruction.
PROMPT = (
    "Outpaint the image: replace the solid grey areas with an empty continuation of the same room "
    "and scene beyond the frame, keeping the existing picture unchanged. Continue the same floor, "
    "the same wall and background and the same soft indoor daylight as the photo, at the same "
    "perspective, scale and brightness. The added area holds no person, no child, no face, no hands "
    "and no figure; keep every person already in the photo exactly as they are, with the same "
    "faces, expressions, hair and clothes."
)
NEGATIVE_PROMPT = (
    "person, child, man, woman, crowd, face, hands, portrait, new person, extra person, "
    "second child, mirrored child, reflected child, reflection of a person, different face, "
    "changed expression, deformed, distorted, blurry, low quality, text, watermark"
)

# No-new-people gate: two local detectors over the added panels of every tile
GATE = PersonGate(PROJ_ROOT / "models/photo-animation/ComfyUI/models/ultralytics")

# Server
SETUP = Setup.load(PROJ_ROOT / "tmp/workflows/image-extender/setup-extender.json")
SERVER = ComfyServer(SETUP)  # starts ComfyUI on the pinned GPU, port 8189

rprint(
    f"[bold medium_purple]Configuration[/bold medium_purple]\n"
    f"[dim]{'-' * 44}[/dim]\n"
    f"[bold]Frames[/bold]\n"
    f"  Width [light_sea_green]1280 px[/light_sea_green] [dim]the height follows the aspect[/dim]\n"
    f"  4:3  [light_sea_green]1280 x 960 px[/light_sea_green] [dim](278 px left, 279 px right, 43.5% new)[/dim]\n"
    f"  16:9 [light_sea_green]1280 x 720 px[/light_sea_green] [dim](369 px each side, 57.7% new)[/dim]\n"
    f"  Photo [light_sea_green]scaled to the frame height[/light_sea_green] [dim]723 x 960 and 542 x 720[/dim]\n\n"
    f"[bold]Models[/bold]\n"
    f"  Extenders [cyan]{len(registry.EXTENDERS)}[/cyan] [dim]{', '.join(m.label for m in registry.EXTENDERS)}[/dim]\n"
    f"  Upscaling [dim]deferred by the order of 2026-10-25 06:00[/dim]\n"
    f"  Seed [dark_sea_green]{SEED}[/dark_sea_green]\n\n"
    f"[bold]Device[/bold]\n"
    f"  Server [green]{SERVER.stats['devices'][0]['name']}[/green]\n"
    f"  Port [light_sea_green]{SETUP.port}[/light_sea_green]"
)


## Data Loading

Reads the working copy for its size and states the two frames the geometry produces. The photo is portrait (0.7527), so the 1280 px frame scales it to 960 or 720 px high and needs 278 px on the left and 279 on the right at 4:3, and 369 on each side at 16:9. The original in `data/external/` and the working copy in `data/interim/` are never modified.


In [ ]:
with PilImage.open(SOURCE) as opened:
    source_size = opened.size
assert source_size == SOURCE_SIZE, f"{SOURCE} is {source_size}, expected {SOURCE_SIZE}"

frame_table = Table(title="Frames", box=box.SIMPLE_HEAVY)
for column in ("Aspect", "Frame", "Scaled photo", "Side pad", "New scene"):
    frame_table.add_column(column, style="grey70" if column == "Aspect" else "white")
for name, ratio in ASPECTS.items():
    geometry = composite.Geometry.for_aspect(source_size, ratio)
    photo = geometry.photo_size(source_size)
    new_area = 1 - photo[0] / geometry.width
    frame_table.add_row(
        f"{name} ({ratio:.4f})",
        f"[light_sea_green]{geometry.width} x {geometry.height}[/light_sea_green]",
        f"{photo[0]} x {photo[1]}",
        f"{geometry.pad_left} left, {geometry.pad_right} right",
        f"[dark_sea_green]{new_area:.1%}[/dark_sea_green]",
    )
rprint(frame_table)


## Outpainting models

Every extender runs twice on the same source, prompt and seed: once to 4:3 and once to 16:9. The module scales the photo to the frame height, pads the sides with flat grey, runs the model, tone-matches the render to the scaled photo's own colour statistics, pastes that scaled photo back unchanged and writes the PNG with a sidecar naming the model, weights, licence, prompt, negative prompt, seed, size, seconds and peak VRAM.

A model that reaches its own size limit, or runs out of memory, is retried at 71 and then 50 percent of the target; the largest size that fits is recorded. One model failing records the failure and carries on with the others.

No tile here is accepted on trust. After the photo is pasted back, two local detectors run over the frame: a face or person box wholly inside an added panel is a stranger and rejects the tile, while a face box crossing the photo's edge is a continuation of someone the photo already shows and is recorded without rejecting the tile. A rejected tile is re-rendered with the next seed, 42 to 47; the first clean tile is kept, and a tile that never passes is marked failed in its sidecar with its stranger and continuation boxes and then shown in the contact sheet with its failed label.


In [ ]:
outpaint_jobs = [(m, aspect) for m in registry.EXTENDERS for aspect in ASPECTS]
extended_results = []
outpaint_failures = []

with Progress(
    SpinnerColumn(),
    TextColumn("{task.description}"),
    BarColumn(),
    TextColumn("{task.completed}/{task.total}"),
    TimeElapsedColumn(),
) as progress:
    task = progress.add_task("Outpainting", total=len(outpaint_jobs))
    for model, aspect in outpaint_jobs:
        progress.update(task, description=f"[cyan]{model.label}[/cyan] [dim]{aspect}[/dim]")
        output = EXTENDED_DIR / aspect / model.name / "IMG_8564.png"
        try:
            extended_results.append(
                extend(SERVER, model, SOURCE, aspect, output, PROMPT, NEGATIVE_PROMPT, GATE, SEED)
            )
        except Exception as error:  # one model failing must not stop the others
            outpaint_failures.append(
                {"stage": "outpaint", "model": model.name, "aspect": aspect, "error": str(error)}
            )
            rprint(f"[red]{model.name} {aspect} failed: {error}[/red]")
        progress.advance(task)

rprint(
    f"[bold light_sea_green]{len(extended_results)}[/bold light_sea_green] extended, "
    f"[bold red]{len(outpaint_failures)}[/bold red] failed"
)

## Outpainting results

One row per rendered frame: the size the model actually rendered, the wall seconds and the peak VRAM above the idle baseline, and two continuity measures of the new side panels. The seam ratio divides the grey-level jump across the photo's border by the mean jump inside the photo; 1.0 means the fill continues the photo as smoothly as the photo continues itself. The panel detail is the standard deviation of the two new panels.

In [ ]:
def seam_row(result) -> dict:
    path = PROJ_ROOT / result.image
    with PilImage.open(path) as opened:
        frame = opened.convert("RGB")
    geometry = composite.Geometry(**result.geometry)
    photo_size = geometry.photo_size((result.photo_width, result.photo_height))
    return composite.seam_metrics(frame, geometry.origin(photo_size), photo_size)


seam_rows = {(r.extender, r.aspect): seam_row(r) for r in extended_results}
result_table = Table(title="Extended frames", box=box.SIMPLE_HEAVY)
for column in ("Extender", "Aspect", "Render", "Output", "Seconds", "Peak GB", "Seam", "Panel detail"):
    result_table.add_column(column, style="grey70" if column == "Extender" else "white")
for result in extended_results:
    metrics = seam_rows[(result.extender, result.aspect)]
    result_table.add_row(
        result.extender,
        result.aspect,
        f"{result.render_width} x {result.render_height}",
        f"{result.width} x {result.height}",
        f"{result.seconds:.1f}",
        f"{result.peak_vram_gb:.1f}",
        f"{metrics['seam']:.2f}",
        f"{metrics['panel_std']:.1f}",
    )
rprint(result_table)


## No-new-people gate

One row per extended frame: the seed that was kept, how many seeds were tried, which seeds the gate rejected and how many detector boxes they held. A tile is rejected when a face or person box lies wholly inside an added panel; a face box crossing the photo's edge is a continuation of someone the photo already shows and is recorded without rejecting the tile. The sidecar of every tile carries the same record, with each box's coordinates and its fraction inside the panels.

A frame that failed every seed is still written, marked failed in its sidecar and shown in the contact sheet with its failed label, so the failure is visible rather than hidden.


In [ ]:
gate_table = Table(title="No-new-people gate", box=box.SIMPLE_HEAVY)
for column in ("Extender", "Aspect", "Seed used", "Tries", "Rejected seeds", "Boxes", "Passed"):
    gate_table.add_column(column, style="grey70" if column == "Extender" else "white")
gate_failures = []
for result in extended_results:
    gate = result.gate
    rejected = gate.get("rejected", [])
    boxes = sum(len(item["violations"]) for item in rejected)
    if not gate.get("passed"):
        gate_failures.append((result.extender, result.aspect))
    gate_table.add_row(
        result.extender,
        result.aspect,
        str(gate.get("seed")),
        str(gate.get("attempts")),
        ", ".join(str(item["seed"]) for item in rejected) or "none",
        str(boxes),
        "[light_sea_green]yes[/light_sea_green]" if gate.get("passed") else "[red]FAILED[/red]",
    )
rprint(gate_table)
rprint(
    f"[bold red]{len(gate_failures)}[/bold red] extended frames failed the gate and are "
    "marked failed in their sidecars"
)


## Upscaling (deferred)

Upscaling is deferred by the Star Colonel's order of 2026-10-25 06:00: the frames are 1280 wide so the render time drops sharply, and they can be raised later. The upscaler code and its tests stay in `src/henryk_media/imaging/`; this bench run performs extension only, so the notebook has no combined images and no upscaler column.


## Evaluation

Counts, failures and the per-model answer to the harder frame: whether the scene degrades at 16:9 compared with the same model at 4:3. Degradation is recorded as a rule, not a judgement: the 16:9 seam ratio more than twice the 4:3 value, or the 16:9 panel detail below 60 percent of the 4:3 value.

The metrics are proxies. They measure colour continuity and structure across the seam, not whether the room looks right. The contact sheets are the record the Star Colonel judges; the numbers here flag where to look first.

In [ ]:
degradation = []
for model in registry.EXTENDERS:
    four = seam_rows.get((model.name, "4x3"))
    nine = seam_rows.get((model.name, "16x9"))
    if four is None or nine is None:
        degradation.append((model.label, "not rendered", "-", "-", "-"))
        continue
    seam_ratio = nine["seam"] / max(four["seam"], 1e-6)
    detail_ratio = nine["panel_std"] / max(four["panel_std"], 1e-6)
    verdict = "degrades" if seam_ratio > 2.0 or detail_ratio < 0.6 else "holds"
    degradation.append(
        (model.label, verdict, f"{seam_ratio:.2f}", f"{detail_ratio:.2f}", f"{nine['seam']:.2f}")
    )

degradation_table = Table(title="16:9 against 4:3", box=box.SIMPLE_HEAVY)
for column in ("Extender", "Scene at 16:9", "Seam x", "Detail x", "16:9 seam"):
    degradation_table.add_column(column, style="grey70" if column == "Extender" else "white")
for row in degradation:
    degradation_table.add_row(*row)
rprint(degradation_table)

rprint(
    f"\n[bold]{len(extended_results)}[/bold] extended frames, "
    f"[bold red]{len(outpaint_failures)}[/bold red] failures"
)
for failure in outpaint_failures:
    rprint(f"  [red]{failure['stage']}[/red] {failure['model']}: {failure['error'][:160]}")


## Findings

The verdicts above, stated per model. The mechanism is the same for every edit-style model: it re-renders the whole canvas from a grey-padded input, so the larger the new area, the less of the photo remains as context and the more the panels rely on the prompt. A mask model is insulated from that by the ControlNet, which is why its number is the one to compare against.

In [ ]:
for label, verdict, seam_ratio, detail_ratio, seam_nine in degradation:
    if verdict == "not rendered":
        rprint(f"[yellow]{label}[/yellow]: not rendered, no verdict")
        continue
    colour = "light_sea_green" if verdict == "holds" else "red"
    rprint(
        f"[bold]{label}[/bold]: [{colour}]{verdict}[/{colour}] at 16:9 "
        f"[dim](seam x{seam_ratio}, detail x{detail_ratio}, 16:9 seam {seam_nine})[/dim]"
    )

## Comparison

The last step: one contact sheet per aspect, one tile per extender with the source photo in the top-left corner. Every tile is the same size and carries the extender's label. A tile whose render failed is marked failed and keeps its place, so the sheet shows every model that was tried.

Every tile is shown as an image, including a tile that failed the gate; its label states the verdict and the stranger and continuation counts, so the sheet is complete and the failure is visible. The sheets are displayed by link. No image data of the child is embedded in this notebook.


In [ ]:
def verdict_tile(path, label, loader) -> grids.Tile:
    """Return the tile with its gate verdict in the label; a failed tile keeps its image."""
    record = loader(path)
    if record is None:
        return grids.Tile(path, f"{label} - no record")
    gate = record.gate
    strangers = kept_strangers(gate)
    continuations = gate.get("continuations") or []
    counts = [
        f"{count} {noun}" + ("" if count == 1 else "s")
        for count, noun in ((len(strangers), "stranger"), (len(continuations), "continuation"))
        if count
    ]
    verdict = "passed" if gate.get("passed") else "FAILED"
    label = f"{label} - {verdict}" + (f", {', '.join(counts)}" if counts else "")
    return grids.Tile(path, label)


grid_paths = {}

for aspect in ASPECTS:
    # The source photo sits in the top-left corner, then one tile per extender.
    grid = [[grids.Tile(SOURCE, "source photo")]]
    grid[0].extend(
        verdict_tile(
            EXTENDED_DIR / aspect / model.name / "IMG_8564.png", model.label, load_extend
        )
        for model in registry.EXTENDERS
    )
    grid_paths[aspect] = grids.contact_sheet(grid, EXTENDED_DIR / f"grid-{aspect}.png")
    display(Markdown(f"### {aspect} - {len(registry.EXTENDERS)} extenders"))
    display(Image(url=os.path.relpath(grid_paths[aspect]), width=1400))
